In [1]:
# ==========================================================
# Imports
# ==========================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from catboost import CatBoostClassifier

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss

pd.set_option("display.max_columns", None)

In [2]:
# ==========================================================
# Load Dataset
# ==========================================================

DATA = "/kaggle/input/notebooks/shri7ul/03-feature-engineering-ipynb/master_feature_engineered.parquet"

master = pd.read_parquet(DATA)

TARGET = "is_correct"

DROP_COLUMNS = [

    "response_id",
    "session_id",

    "learning_objective",
    "learning_objective_id",

    "transcript",
    "student_text",
    "tutor_text",
    "background_text",

    TARGET,
]

X = master.drop(columns=DROP_COLUMNS)

y = master[TARGET].astype(int)

cat_cols = X.select_dtypes(include="object").columns.tolist()

for col in cat_cols:
    X[col] = X[col].astype("category").cat.codes

print(X.shape)

(35072, 113)


In [3]:
# ==========================================================
# Tune Learning Rate
# ==========================================================

learning_rates = [
    0.20,
    0.10,
    0.05,
    0.03,
    0.02,
    0.01,
]

results = []

for lr in learning_rates:

    print("="*60)
    print(f"learning_rate = {lr}")
    print("="*60)

    kf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    oof = np.zeros(len(X))

    for train_idx, valid_idx in kf.split(X, y):

        X_train = X.iloc[train_idx]
        X_valid = X.iloc[valid_idx]

        y_train = y.iloc[train_idx]
        y_valid = y.iloc[valid_idx]

        model = CatBoostClassifier(

            iterations=5000,

            learning_rate=lr,

            depth=6,

            loss_function="Logloss",

            eval_metric="Logloss",

            random_seed=42,

            od_type="Iter",

            od_wait=300,

            verbose=False,
        )

        model.fit(

            X_train,
            y_train,

            eval_set=(X_valid, y_valid),

            use_best_model=True,

            verbose=False,
        )

        pred = model.predict_proba(X_valid)[:,1]

        oof[valid_idx] = pred

    score = log_loss(y, oof)

    results.append({
        "learning_rate": lr,
        "logloss": score
    })

    print(f"OOF LogLoss : {score:.6f}")
    print()

results = (
    pd.DataFrame(results)
    .sort_values("logloss")
    .reset_index(drop=True)
)

display(results)

learning_rate = 0.2
OOF LogLoss : 0.551469

learning_rate = 0.1
OOF LogLoss : 0.548243

learning_rate = 0.05
OOF LogLoss : 0.546569

learning_rate = 0.03
OOF LogLoss : 0.545532

learning_rate = 0.02
OOF LogLoss : 0.545710

learning_rate = 0.01
OOF LogLoss : 0.544650



,learning_rate,logloss
0,0.01,0.544650
1,0.03,0.545532
2,0.02,0.545710
3,0.05,0.546569
4,0.10,0.548243
5,0.20,0.551469


In [4]:
# ==========================================================
# Tune Depth
# ==========================================================

depths = [4, 5, 6, 7, 8, 9, 10]

results = []

for depth in depths:

    print("="*60)
    print(f"depth = {depth}")
    print("="*60)

    kf = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42,
    )

    oof = np.zeros(len(X))

    for train_idx, valid_idx in kf.split(X, y):

        X_train = X.iloc[train_idx]
        X_valid = X.iloc[valid_idx]

        y_train = y.iloc[train_idx]
        y_valid = y.iloc[valid_idx]

        model = CatBoostClassifier(

            iterations=5000,

            learning_rate=0.01,
            depth=depth,

            loss_function="Logloss",
            eval_metric="Logloss",

            random_seed=42,

            od_type="Iter",
            od_wait=300,

            use_best_model=True,

            verbose=False,
        )

        model.fit(
            X_train,
            y_train,
            eval_set=(X_valid, y_valid),
            verbose=False,
        )

        pred = model.predict_proba(X_valid)[:, 1]

        oof[valid_idx] = pred

    score = log_loss(y, oof)

    results.append({
        "depth": depth,
        "logloss": score,
    })

    print(f"OOF LogLoss : {score:.6f}")
    print()

results = (
    pd.DataFrame(results)
    .sort_values("logloss")
    .reset_index(drop=True)
)

display(results)

depth = 4
OOF LogLoss : 0.547320

depth = 5
OOF LogLoss : 0.545637

depth = 6
OOF LogLoss : 0.544650

depth = 7
OOF LogLoss : 0.544381

depth = 8
OOF LogLoss : 0.544124

depth = 9
OOF LogLoss : 0.544403

depth = 10
OOF LogLoss : 0.545322



,depth,logloss
0,8,0.544124
1,7,0.544381
2,9,0.544403
3,6,0.544650
4,10,0.545322
5,5,0.545637
6,4,0.547320
